# 01 · Phase 1 data audit (FETAL_PLANES_DB → 4 classes)
Reproduces the published Phase 1 split exactly (`--verify-phase1`), then builds the v2 splits with perceptual
near-duplicate removal (`group_shuffle`) and the class-stratified patient split (`stratified_group`).
Outputs: `$FETALVISION_DATA/fetal_planes/processed/<strategy>/` and `results/phase1/data_audit/`.

## 0. Environment (Colab)
Clone the repository, install pinned requirements, set `FETALVISION_DATA`, mount Google Drive and unzip the Zenodo
archive (`FETAL_PLANES_ZENODO.zip`, record 3904280, CC BY 4.0) into `$FETALVISION_DATA/fetal_planes/raw/`.
All logic lives in the `fetalvision` package; this notebook only calls its CLIs.

In [ ]:
import os, subprocess, pathlib
if not pathlib.Path('FetalVision').exists():
    !git clone https://github.com/lamooooo1418/FetalVision.git
%cd FetalVision
!pip -q install -r requirements.txt -e .
os.environ['FETALVISION_DATA'] = '/content/data'
pathlib.Path('/content/data/fetal_planes/raw').mkdir(parents=True, exist_ok=True)
print('FETALVISION_DATA =', os.environ['FETALVISION_DATA'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/fetal planes/FETAL_PLANES_ZENODO.zip'   # adjust to where the Zenodo zip lives on your Drive
!test -f "$ZIP" && unzip -q -n "$ZIP" -d "$FETALVISION_DATA/fetal_planes/raw/extracted" || echo "zip not found: download https://zenodo.org/records/3904280"
!ls "$FETALVISION_DATA/fetal_planes/raw/extracted" | head

## 1. Exact reproduction of the Phase 1 split (near-duplicate removal OFF)

In [ ]:
!python -m fetalvision.data.prepare_planes --config configs/phase1_data.yaml --verify-phase1

## 2. v2 split: group_shuffle + near-duplicate removal (pHash ≤ 4 AND dHash ≤ 6 bits)

In [ ]:
!python -m fetalvision.data.prepare_planes --config configs/phase1_data.yaml --split-strategy group_shuffle --near-dup on

## 3. v2 split: StratifiedGroupKFold (patient-grouped, class-stratified 70/15/15)

In [ ]:
!python -m fetalvision.data.prepare_planes --config configs/phase1_data.yaml --split-strategy stratified_group --near-dup on

## 4. Audit summary

In [ ]:
import json
for name in ['group_shuffle_nodedup', 'group_shuffle', 'stratified_group']:
    a = json.load(open(f'results/phase1/data_audit/audit_{name}.json'))
    print('='*20, name, '='*20)
    print('images/patients:', a['split']['n_images'], a['split']['n_patients'], '| per split:', a['split']['totals'], a['split']['patients'])
    print('counts:', json.dumps(a['split']['counts']))
    print('removed:', {k: v for k, v in a['removed'].items() if isinstance(v, int)})
    nd = a['near_duplicates']; print('near-dups within/across:', nd['within_patient'], nd['across_patients'], '| removed:', nd['removed_within_patient'], nd['removed_across_patients'])
    print('overlap checks:', a['overlap_checks'])
    print('label audit:', a['label_audit_round1'])
    if 'phase1_reproduction' in a: print('PHASE 1 REPRODUCED:', a['phase1_reproduction']['match'])

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob('results/phase1/data_audit/figures/*.png')):
    print(f); display(Image(filename=f, width=800))

## 5. Second-round label-audit sample (25/class from val+test + every error of the kept Phase 1 model)

In [ ]:
import pandas as pd
s = pd.read_csv('results/phase1/data_audit/label_audit_sample.csv'); print(len(s), 'rows'); print(s.selection.value_counts()); s.head()